# Chapitre 15 · Passer à l'échelle : la vraie chose (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du notebook du
chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le minimum pour que tout s'exécute de façon autonome : les imports et l'utilitaire
`lance_torchrun` du notebook du chapitre (2 workers CPU, backend gloo).

In [ ]:
import os
import sys
import subprocess
import textwrap

import torch

# L'utilitaire du notebook du chapitre : lancer un script en N workers via torchrun.
def lance_torchrun(script, nproc=2, port=29600, args=None):
    cmd = [sys.executable, "-m", "torch.distributed.run",
           f"--nproc_per_node={nproc}", "--nnodes=1", f"--master_port={port}", script]
    if args:
        cmd += args
    env = dict(os.environ, OMP_NUM_THREADS="1")
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=300)
    lignes = (r.stdout + r.stderr).splitlines()
    return r.returncode, lignes

print("PyTorch", torch.__version__, "| prêt.")

### Exercice 1 · La moyenne des gradients — niveau ●

L'all-reduce, réduit à son arithmétique : chaque worker envoie son gradient, tous
reçoivent la somme, qu'on divise par le nombre de workers. Écris cette opération en
pur PyTorch, sans processus ni gloo : une fonction qui reçoit la liste des gradients
locaux et renvoie ce que chaque worker porte APRÈS l'all-reduce.

In [ ]:
def allreduce_moyenne(grads):
    total = grads[0].clone()
    for g in grads[1:]:
        total = total + g              # la SOMME des gradients de tous les workers
    moyenne = total / len(grads)       # ... transformée en MOYENNE
    return [moyenne.clone() for _ in grads]   # chacun repart avec la même valeur

In [ ]:
# Validation : all-reduce = somme / nombre de workers.
apres = allreduce_moyenne([torch.tensor([2.0211]), torch.tensor([-0.6770])])
assert len(apres) == 2, "chaque worker doit repartir avec un gradient"
assert all(torch.allclose(a, torch.tensor([0.67205])) for a in apres), "chacun doit porter la moyenne"
apres3 = allreduce_moyenne([torch.tensor([3.0]), torch.tensor([0.0]), torch.tensor([0.0])])
assert all(torch.allclose(a, torch.tensor([1.0])) for a in apres3), "ça doit marcher pour 3 workers aussi"
print(f"All-reduce en petit OK : +2.0211 et -0.6770 -> {apres[0].item():+.5f} partout")

### Exercice 2 · L'all-reduce à la main, en vrai — niveau ●●

Maintenant avec 2 vrais processus qui communiquent par gloo. Le script ci-dessous est
celui de la section 2.2, MOINS les deux lignes qui soudent la grappe. Remplace les
`# TODO(toi)` par l'all-reduce à la main (additionner les gradients de tous les
workers, puis diviser par leur nombre), supprime le `raise`, puis lance la validation.
Essaie de mémoire avant de remonter à la section 2.2.

In [ ]:
EXO_ALLREDUCE = textwrap.dedent("""
    import os
    import torch, torch.distributed as dist
    import torch.nn as nn, torch.nn.functional as F

    rank = int(os.environ["RANK"]); world = int(os.environ["WORLD_SIZE"])
    dist.init_process_group("gloo", rank=rank, world_size=world)

    torch.manual_seed(42)                 # MÊME init -> répliques identiques
    model = nn.Linear(4, 1, bias=False)
    torch.manual_seed(100 + rank)         # données PROPRES à chaque worker
    x, y = torch.randn(8, 4), torch.randn(8, 1)
    F.mse_loss(model(x), y).backward()
    g = model.weight.grad

    before = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), before, dst=0)
    dist.all_reduce(g, op=dist.ReduceOp.SUM)   # chaque worker reçoit la SOMME
    g /= world                                  # ... transformée en MOYENNE
    after = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), after, dst=0)

    if rank == 0:
        for r in range(world):
            print(f"worker {r} | AVANT all-reduce : grad = {before[r].item():+.4f}")
        for r in range(world):
            print(f"worker {r} | APRES all-reduce : grad = {after[r].item():+.4f}")
    dist.destroy_process_group()
""")
with open("exo_allreduce.py", "w") as f:
    f.write(EXO_ALLREDUCE)
print("exo_allreduce.py écrit.")

In [ ]:
# Validation : l'all-reduce à la main, exécuté par 2 vrais processus.
code_exo, lignes_exo = lance_torchrun("exo_allreduce.py", nproc=2, port=29611)
assert code_exo == 0, "le script a échoué : as-tu remplacé les TODO (et supprimé le raise) ?"
avant = [float(l.rsplit("=", 1)[1]) for l in lignes_exo if "AVANT all-reduce" in l]
apres = [float(l.rsplit("=", 1)[1]) for l in lignes_exo if "APRES all-reduce" in l]
assert len(avant) == 2 and len(apres) == 2, "il manque des lignes de sortie"
assert abs(apres[0] - apres[1]) < 1e-6, "les deux workers doivent porter la MÊME valeur"
assert abs(apres[0] - sum(avant) / 2) < 1e-3, "la valeur portée doit être la MOYENNE des gradients"
print(f"All-reduce OK : {avant[0]:+.4f} et {avant[1]:+.4f} -> {apres[0]:+.4f} sur les deux workers")

### Exercice 3 · Chacun sa part : le DistributedSampler — niveau ●●

Une grappe de 2 workers, un dataset de 32 exemples. Construis la part de chaque worker
avec `DistributedSampler` (mélange activé), et vérifie la promesse de la section 2.5 :
deux parts de 16, sans recouvrement, qui couvrent tout le dataset.

In [ ]:
from torch.utils.data import TensorDataset, DistributedSampler

dataset_exo = TensorDataset(torch.arange(32))

sampler_0 = DistributedSampler(dataset_exo, num_replicas=2, rank=0, shuffle=True)
sampler_1 = DistributedSampler(dataset_exo, num_replicas=2, rank=1, shuffle=True)
part_0 = list(sampler_0)
part_1 = list(sampler_1)
print("worker 0 :", part_0)
print("worker 1 :", part_1)

In [ ]:
# Validation : découpage sans recouvrement.
assert sorted(map(len, [part_0, part_1])) == [16, 16], "deux parts de 16 exemples chacune"
assert set(part_0) & set(part_1) == set(), "aucun exemple ne doit être vu par les deux workers"
assert set(part_0) | set(part_1) == set(range(32)), "aucun exemple ne doit être oublié"
print("DistributedSampler OK : 32 exemples, 2 parts de 16, sans recouvrement")

### Exercice 4 · Le point Chinchilla-optimal — niveau ●●●

Combine deux règles que tu connais : le coût d'entraînement C ≈ 6 · N · D FLOPs
(chapitre 14) et l'équilibre Chinchilla D = 20 · N (20 tokens par paramètre). Pour un
budget C fixé, résous les deux équations : quelle taille de modèle N, et combien de
tokens D ? (Il te faut une racine carrée : `math.sqrt`.)

In [ ]:
import math

def chinchilla_optimal(budget_flops):
    n_params = math.sqrt(budget_flops / 120)   # C = 6 * N * (20 N) = 120 N**2
    n_tokens = 20 * n_params                   # la règle des 20 tokens par paramètre
    return n_params, n_tokens

In [ ]:
# Validation : le point Chinchilla-optimal.
n, d = chinchilla_optimal(1.2e20)
assert abs(n - 1e9) / 1e9 < 1e-6, f"pour 1.2e20 FLOPs, N attendu ~1e9, obtenu {n:.3e}"
assert abs(d - 2e10) / 2e10 < 1e-6, f"pour 1.2e20 FLOPs, D attendu ~2e10, obtenu {d:.3e}"
n2, d2 = chinchilla_optimal(6 * 70e9 * 1.4e12)   # le budget de Chinchilla lui-même
assert abs(d2 / n2 - 20) < 1e-6, "le ratio tokens/paramètres doit valoir 20"
assert abs(n2 - 70e9) / 70e9 < 1e-3, f"N attendu ~70 Md, obtenu {n2:.3e}"
print(f"Chinchilla OK : 1.2e20 FLOPs -> N = {n:.2e} paramètres, D = {d:.2e} tokens")